This Jupyter notebook uses plotly express to build a stacked bar chart to display Traffic Control Devices by Route Type and District in Place for Traffic Incidents in Construction Work Zones on State-maintained Routes 2020-2024.

In [28]:
import os
import sqlite3
import pandas as pd
import plotly.express as px

In [29]:
# Define location of database file
database_path = os.path.abspath('data/crash_data.db')
print(f"Absolute database path: {database_path}")

if not os.path.exists(database_path):
    print(f"Database file not found at {database_path}")
else:
    print(f"Database file found at {database_path}")

Absolute database path: /Users/terid/CodeYou_Capstone/data/crash_data.db
Database file found at /Users/terid/CodeYou_Capstone/data/crash_data.db


The data was grouped by KYTC District to show the variations in the distribution of Interstates and Parkways across the state.

In [30]:
# SQL query to get the consolidated dataset
query = """
SELECT
    rc.Route_Type,
    c.TrafficControl,
    cd.KYTC_District_Number AS District,
    COUNT(*) AS Count
FROM
    ksp_controls c
JOIN
    Roadway_Characteristics_API rc
    ON c.IncidentID = rc.IncidentID
JOIN
    county_district_lut cd
    ON rc.County_Name = cd.Cnty_Name_PC
WHERE
    rc.Government_Level = 'State Maintained Roads'
GROUP BY
    rc.Route_Type,
    c.TrafficControl,
    District
ORDER BY
    rc.Route_Type,
    c.TrafficControl,
    District;
"""

# Execute the query and load the data into a pandas DataFrame
with sqlite3.connect(database_path) as conn:
    df = pd.read_sql_query(query, conn)

# Specify the order of route types
category_order = ['I', 'PKWY', 'US', 'KY']

In [31]:
# Create a bar chart with Plotly
fig = px.bar(
    df,
    x='Route_Type',
    y='Count',
    color='TrafficControl',
    facet_col='District',
    title='Traffic Control Devices by Route Type and District in Place for Traffic Incidents in Construction Work Zones on State-maintained Routes 2020-2024',
    labels={'Route_Type': 'Route Type', 'Count': 'Device Count', 'TrafficControl': 'Traffic Control Device', 'District': 'District'},
    category_orders={'Route_Type': category_order},
    height=800

)

# Show the plot
fig.show()